In [ ]:
from pathlib import Path
import sys
import os.path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json
from scipy.stats import f_oneway, gaussian_kde, mannwhitneyu, pearsonr, spearmanr
import seaborn as sns

from tqdm.notebook import tqdm
from pymodulon.compare import compare_ica
from pymodulon.core import IcaData
from pymodulon.io import load_json_model, save_to_json
from pymodulon.plotting import *

from pymodulon.util import explained_variance
from pymodulon.imodulondb import imdb_gene_presence
sns.set_style('whitegrid')

In [ ]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['svg.fonttype'] = 'none'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rcParams['font.family'] = 'sans-serif'
sns.set_style('ticks')
matplotlib.rcParams['text.color'] = '#000000'
matplotlib.rcParams['axes.labelcolor'] = '#000000'
matplotlib.rcParams['xtick.color'] = '#000000'
matplotlib.rcParams['ytick.color'] = '#000000'

## Load the data from ./Data/ folder

In [ ]:
ica_data = load_json_model('../../modulome_sc.json')

In [ ]:
Gene_Presence_Matrix = pd.DataFrame(index = ica_data.M.index, columns = ica_data.imodulon_names).fillna(False)

for im in ica_data.imodulon_names:
    genes = ica_data.view_imodulon(im).index
    Gene_Presence_Matrix.loc[genes, im] = True

In [ ]:
X = ica_data.X

In [ ]:
trn = pd.read_csv('../../data/external/trn.csv', index_col=0).dropna()
trn = trn[["reg","gene_name", "gene_id", "effect"]]
trn = trn.rename({"reg":"regulator"}, axis=1)


count = 0
genes_in_regulons = trn["gene_id"].unique()
for gene in X.index:
    if gene in genes_in_regulons:
        count+=1
print("Number of genes with annotated regulators: ",count)
print("Number of unique regulators:", len(trn.regulator.unique()))
#filter trn to only include genes which were aligned to
display(trn.head(), trn.shape)

### PRECISE-1K: X M A log_tpm gene_presence_matrix

In [ ]:
M = ica_data.M

In [ ]:
A = ica_data.A

In [ ]:
X = ica_data.X

In [ ]:
log_tpm = ica_data.log_tpm

## Get the top exp var iM for each gene

In [ ]:
def calculate_explained_variance(M_matrix, A_matrix, log_tpm):
    # Check if inputs are DataFrames and convert to numpy arrays if needed
    if isinstance(M_matrix, pd.DataFrame) or isinstance(M_matrix, pd.Series):
        M_matrix = M_matrix.values.flatten()
    
    if isinstance(A_matrix, pd.DataFrame):
        A_matrix = A_matrix.values
    
    if isinstance(log_tpm, pd.DataFrame) or isinstance(log_tpm, pd.Series):
        log_tpm = log_tpm.values.flatten()
    
    # Compute the predicted values
    predicted_values = np.dot(M_matrix, A_matrix)  # M_matrix is 1xM, A_matrix is MxN, result is 1xN

    # Calculate the baseline (mean-centered log_tpm)
    baseline = log_tpm - np.mean(log_tpm)  # mean-center the log_tpm

    # Calculate the residual variance
    residuals = log_tpm - predicted_values
    residual_variance = np.sum((residuals - np.mean(residuals)) ** 2)

    # Calculate the total variance
    total_variance = np.sum((log_tpm - np.mean(log_tpm)) ** 2)

    # Compute the explained variance (R^2)
    explained_variance = 1 - (residual_variance / total_variance)
    
    return explained_variance

In [ ]:
def compute_explained_variance_change(M, A, X, Gene_Presence_Matrix, gene):
    M = M.loc[[gene]]
    X = X.loc[[gene]]

    # Initialize a dictionary to store the explained variance changes
    exp_var_changes = {}

    # Iterate through each row of A
    for row_name in A.index:
        # Select the corresponding row from A
        modified_A = A.loc[[row_name]]

        # Select the corresponding column from M and X
        modified_M = M[[row_name]]
        modified_X = X

        # Recalculate the explained variance
        new_exp_var = calculate_explained_variance(modified_M, modified_A, modified_X)

        # Store the result in the dictionary
        exp_var_changes[row_name] = new_exp_var

    # Convert the dictionary to a DataFrame
    explained_variance_df = pd.DataFrame.from_dict(exp_var_changes, orient='index', columns=['Explained Variance'])
    explained_variance_df.reset_index(inplace=True)
    
    explained_variance_df.columns = ['iModulon', 'Explained Variance']
    explained_variance_df = explained_variance_df.sort_values(by='Explained Variance', ascending=False)
    
    # Add presence information
    explained_variance_df['Presence'] = explained_variance_df['iModulon'].apply(
        lambda x: 'Yes' if Gene_Presence_Matrix.loc[gene, x] else 'No'
    )

    # Select the top 3 iModulons with the highest explained variance
    top_3 = explained_variance_df.head(3)

    # Return the top 3 iModulons and their explained variances
    return (
        top_3.iloc[0]['iModulon'], top_3.iloc[0]['Explained Variance'],
        top_3.iloc[1]['iModulon'] if len(top_3) > 1 else None, top_3.iloc[1]['Explained Variance'] if len(top_3) > 1 else None,
        top_3.iloc[2]['iModulon'] if len(top_3) > 2 else None, top_3.iloc[2]['Explained Variance'] if len(top_3) > 2 else None
    )

In [ ]:
compute_explained_variance_change(M, A, X, Gene_Presence_Matrix, 'YIL101C')

In [ ]:
def explained_variance_all_genes(M_matrix, A_matrix, log_tpm, Gene_Presence_Matrix):
    results = []

    for gene in tqdm(log_tpm.index, desc="Processing genes"):
        try:
            top_1_imodulon, top_1_exp_var, top_2_imodulon, top_2_exp_var, top_3_imodulon, top_3_exp_var = compute_explained_variance_change(
                M_matrix, A_matrix, log_tpm, Gene_Presence_Matrix, gene
            )
            results.append([gene, top_1_exp_var, top_1_imodulon, top_2_exp_var, top_2_imodulon, top_3_exp_var, top_3_imodulon])
        except Exception as e:
            print(f"Error processing gene {gene}: {e}")
            results.append([gene, None, None, None, None, None, None])

    # Convert the results to a DataFrame
    results_df = pd.DataFrame(results, columns=[
        'Gene', 
        'Top_1_Explained_Variance', 'Top_1_iModulon',
        'Top_2_Explained_Variance', 'Top_2_iModulon',
        'Top_3_Explained_Variance', 'Top_3_iModulon'
    ])
    
    return results_df


In [ ]:
top_exp_var = explained_variance_all_genes(M, A, X, Gene_Presence_Matrix)

In [ ]:
top_exp_var.to_csv('../../data/calculations/top_exp_var.csv')

In [ ]:
top_exp_var = pd.read_csv('../../data/calculations/top_exp_var.csv', index_col=0)

In [ ]:
top_exp_var = top_exp_var.set_index('Gene')

In [ ]:
top_exp_var

In [ ]:
top_exp_var['Top_1_Explained_Variance'].hist()

In [ ]:
top_exp_var.nsmallest(10, 'Top_1_Explained_Variance')

In [ ]:
top_exp_var.nlargest(10, 'Top_1_Explained_Variance')

In [ ]:
import pandas as pd
import numpy as np

def explained_variance_genes(X: pd.DataFrame, M: pd.DataFrame, A: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate the explained variance for each gene from the MA decomposition.

    Args:
        X (pd.DataFrame): Original data with genes as rows and samples as columns.
        M (pd.DataFrame): Matrix with genes as rows and components as columns.
        A (pd.DataFrame): Matrix with components as rows and samples as columns.

    Returns:
        pd.DataFrame: DataFrame with "Total_Explained_Variance" as the only column and gene names as row indexes.
    """
    # Reconstruct the matrix using M and A
    reconstructed_X = M @ A

    # Calculate the total variance of the original data for each gene
    total_variance = X.var(axis=1)

    # Calculate the explained variance for each gene
    residual = X - reconstructed_X
    residual_variance = residual.var(axis=1)

    explained_variance = 1 - (residual_variance / total_variance)
    # explained_variance[explained_variance < 0] = 0

    # Create a DataFrame with Total_Explained_Variance
    explained_variance_df = pd.DataFrame(
        explained_variance, columns=["Total_Explained_Variance"]
    )

    return explained_variance_df

In [ ]:
def compute_total_explained_variance(M_matrix, A_matrix, log_tpm):
    """
    Computes the total explained variance (R^2) for each gene.
    
    Parameters:
    -----------
    M_matrix : pd.DataFrame
        A DataFrame with genes as rows and iModulons as columns.
    A_matrix : pd.DataFrame
        A DataFrame with iModulons as rows and condition/sample indices as columns.
    log_tpm : pd.DataFrame
        A DataFrame with genes as rows and the same condition/sample indices as columns.
    
    Returns:
    --------
    pd.DataFrame
        A DataFrame indexed by gene, with one column: 'Total_Explained_Variance'.
    """
    results = {}

    for gene in M_matrix.index:
        try:
            M_row = M_matrix.loc[[gene]]
            log_row = log_tpm.loc[[gene]]
            total_ev = calculate_explained_variance(M_row, A_matrix, log_row)
            results[gene] = total_ev
        except Exception as e:
            print(f"Error processing gene {gene}: {e}")
            results[gene] = None

    results_df = pd.DataFrame.from_dict(results, orient='index', columns=['Total_Explained_Variance'])
    return results_df

In [ ]:
total_exp_var = compute_total_explained_variance(M, A, X)

In [ ]:
total_exp_var.sort_values('Total_Explained_Variance')

In [ ]:
total_exp_var['Total_Explained_Variance'].min()

In [ ]:
total_exp_var['Total_Explained_Variance'].hist()

## Number of Regulators

In [ ]:
trn = pd.read_csv('../../data/external/trn_full.csv', index_col=0).dropna()
trn = trn[["reg","gene_name", "gene_id", "effect"]]
trn = trn.rename({"reg":"regulator"}, axis=1)


count = 0
genes_in_regulons = trn["gene_id"].unique()
for gene in X.index:
    if gene in genes_in_regulons:
        count+=1
print("Number of genes with annotated regulators: ",count)

trn = trn[trn.gene_id.isin(ica_data.gene_table.index)]

In [ ]:
Regulon_Number = pd.DataFrame(index=X.index)
Regulon_Number['num_regulators'] = 0

for gene in X.index:
    num_regs = len(trn[trn.gene_id == gene].regulator.unique())
    Regulon_Number.loc[gene, 'num_regulators'] = num_regs

In [ ]:
Regulon_Number

In [ ]:
Regulon_Number['num_regulators'] = Regulon_Number['num_regulators'].astype('int')

In [ ]:
Regulon_Number['num_regulators']

## iM count

In [ ]:
num_iM = pd.DataFrame( Gene_Presence_Matrix.sum(axis=1), columns=['num_iM'])

In [ ]:
num_iM

## MAD

In [ ]:
def calculate_mad(matrix: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate the Median Absolute Deviation (MAD) for each gene in the given matrix and return it as a DataFrame.
    
    Args:
    - matrix (pd.DataFrame): A DataFrame where rows represent genes and columns represent samples.
    
    Returns:
    - pd.DataFrame: A DataFrame with one column named 'mAD', containing the MAD for each gene.
    """
    # Compute the median of each row (gene)
    medians = matrix.median(axis=1)
    
    # Compute the absolute deviation from the median for each element in the matrix
    deviations = matrix.sub(medians, axis=0).abs()
    
    # Compute the median of these deviations for each gene
    mad = deviations.median(axis=1)
    
    # Create a DataFrame with the MAD values
    mad_df = pd.DataFrame({'MAD': mad})
    
    return mad_df

In [ ]:
MAD = calculate_mad(log_tpm)

In [ ]:
MAD

## Median Expression

In [ ]:
def calculate_median(matrix: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate the Median Absolute Deviation (MAD) for each gene in the given matrix and return it as a DataFrame.
    
    Args:
    - matrix (pd.DataFrame): A DataFrame where rows represent genes and columns represent samples.
    
    Returns:
    - pd.DataFrame: A DataFrame with one column named 'mAD', containing the MAD for each gene.
    """
    # Compute the median of each row (gene)
    medians = matrix.median(axis=1)
    
    # Create a DataFrame with the MAD values
    mad_df = pd.DataFrame({'log_TPM_median': medians})
    
    return mad_df

In [ ]:
median = calculate_median(log_tpm)

In [ ]:
median

## Range of log tpm

In [ ]:
def calculate_range_log_tpm(df):
    # Calculate the range (max - min) for each row (gene)
    df['range_log_tpm'] = df.max(axis=1) - df.min(axis=1)
    
    # Return a new DataFrame with only the 'range_log_tpm' column
    result_df = df[['range_log_tpm']]
    
    return result_df

In [ ]:
range = calculate_range_log_tpm(log_tpm)

In [ ]:
range

## Quantile Range of log tpm

In [ ]:
def calculate_quantile_range(df):
    # Calculate the 95th and 5th quantile for each row (gene)
    q95 = df.quantile(0.95, axis=1)
    q05 = df.quantile(0.05, axis=1)
    
    # Calculate the difference (95th quantile - 5th quantile)
    df['quantile_range_log_tpm'] = q95 - q05
    
    # Return a new DataFrame with only the 'quantile_range' column
    result_df = df[['quantile_range_log_tpm']]
    
    return result_df

In [ ]:
quantile_range = calculate_quantile_range(log_tpm)

In [ ]:
quantile_range

## Merge Dataframes

In [ ]:
merged = pd.concat([total_exp_var, top_exp_var, Regulon_Number, num_iM, MAD, median, range, quantile_range], axis=1)

In [ ]:
merged.to_csv('../../data/calculations/merged.csv')

In [ ]:
merged

In [ ]:
merged.Total_Explained_Variance.sort_values(ascending=False).head(3000)

In [ ]:
totals = merged.Total_Explained_Variance
totals[totals < 0] = 0
totals.hist()

## Plots of Gene Distribution

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
merged

In [ ]:
plotting_data = pd.DataFrame(columns = ['start', 'end', 'flow'])

plotting_data.loc[len(plotting_data)] = ['all_genes', 'well_explained', len(merged[merged.Total_Explained_Variance > .7])]
plotting_data.loc[len(plotting_data)] = ['well_explained', 'dominant_regulation', len(merged[(merged.Total_Explained_Variance > .7) & (merged.Top_1_Explained_Variance > .5)])]
plotting_data.loc[len(plotting_data)] = ['well_explained', 'combined_regulation', len(merged[(merged.Total_Explained_Variance > .7) & (merged.Top_1_Explained_Variance < .5)])]

plotting_data.loc[len(plotting_data)] = ['all_genes', 'poorly_explained', len(merged[merged.Total_Explained_Variance <= .7])]
plotting_data.loc[len(plotting_data)] = ['poorly_explained', 'high_MAD', len(merged[(merged.Total_Explained_Variance <= .7) & (merged.MAD > .5)])]
plotting_data.loc[len(plotting_data)] = ['poorly_explained', 'low_MAD', len(merged[(merged.Total_Explained_Variance <= .7) & (merged.MAD <= .5)])]

In [ ]:
df = plotting_data
# Get unique node labels
all_nodes = list(pd.unique(df[['start', 'end']].values.ravel()))
node_indices = {node: idx for idx, node in enumerate(all_nodes)}

# Map labels to indices
df['source'] = df['start'].map(node_indices)
df['target'] = df['end'].map(node_indices)

# Calculate total flow per node (incoming + outgoing)
flow_in = df.groupby('end')['flow'].sum()
flow_out = df.groupby('start')['flow'].sum()


# Custom labeling logic
node_labels = []
for node in all_nodes:
    if node == 'all_genes':
        label = f"{node} ({int(flow_out.get(node, 0))})"
    else:
        label = f"{node} ({int(flow_in.get(node, 0))})"
    node_labels.append(label)

# Create Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels
    ),
    link=dict(
        source=df['source'],
        target=df['target'],
        value=df['flow']
    ))])

fig.update_layout(title_text="Genes Explained by iModulons", font_size=10)
fig.show()
fig.write_image("images/sankey_diagram.svg")

In [ ]:
#imodulon model performance across exp levels
fig, ax = plt.subplots(figsize=(10, 7))
merged_plot = merged[merged['Total_Explained_Variance'] >= -1].copy()

merged_plot['Expression_Bin'] = pd.cut(
    merged_plot['log_TPM_median'],
    bins=[0, 5, 8, 11, 20],
    labels=['Low\n(0-5)', 'Medium\n(5-8)', 'High\n(8-11)', 'Very High\n(11+)']
)

merged_plot = merged_plot.dropna(subset=['Expression_Bin'])

sns.boxplot(
    data=merged_plot,
    x='Expression_Bin',
    y='Total_Explained_Variance',
    palette=['#4A90D9', '#5BA85A', '#E8A838', '#E94560'],
    width=0.5,
    flierprops=dict(marker='o', markerfacecolor='gray', markersize=3, alpha=0.3),
    ax=ax
)

ax.axhline(y=0.7, color='orange', linestyle='--', linewidth=1.5, label='R²=0.7 threshold')
ax.axhline(y=0.0, color='red', linestyle='--', linewidth=1.5, label='R²=0 threshold')

# add median annotation above each box
for i, bin_label in enumerate(['Low\n(0-5)', 'Medium\n(5-8)', 'High\n(8-11)', 'Very High\n(11+)']):
    subset = merged_plot[merged_plot['Expression_Bin'] == bin_label]
    if len(subset) > 0:
        median = subset['Total_Explained_Variance'].median()
        ax.text(i, median + 0.03, f'median={median:.2f}',
                ha='center', fontsize=9, color='black', fontweight='bold')

ax.set_xlabel('Median Expression Level (log TPM)', fontsize=12)
ax.set_ylabel('Total Explained Variance (R²)', fontsize=12)
ax.set_title('iModulon Model Performance Across Expression Levels', 
             fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('images/supp_fig_boxplot_expression.svg', dpi=300, bbox_inches='tight', format = 'svg',)
plt.show()

In [ ]:
#regulatory complexity
df_plot = merged[(merged['num_regulators'] <= 50) & (merged['Total_Explained_Variance'] >= -1)]

fig, ax = plt.subplots(figsize=(10, 7))
hb = ax.hexbin(df_plot['num_regulators'], df_plot['Total_Explained_Variance'],
               gridsize=40, cmap='Blues', mincnt=1, alpha=0.8)
plt.colorbar(hb, ax=ax, label='Number of genes')
ax.axhline(y=0.7, color='orange', linestyle='--', linewidth=1.5, label='R²=0.7 threshold')
ax.axhline(y=0.0, color='red', linestyle='--', linewidth=1.5, label='R²=0 threshold')

genes_of_interest = {
    'YDR432W (NPL3)': (-0.9, 35),
    'YNL222W (SSU72)': (-0.1, 23),
    'YHL006C (SHU1)': (-0.05, 18),
    'YDL043C (PRP11)': (-0.02, 14)
}
for label, (r2, regs) in genes_of_interest.items():
    ax.scatter(regs, r2, color='#E94560', s=100, zorder=5)
    ax.annotate(label, (regs, r2), textcoords='offset points',
                xytext=(5, 5), fontsize=9, color='#E94560', fontweight='bold')

ax.set_xlabel('Number of Transcription Regulators', fontsize=12)
ax.set_ylabel('Total Explained Variance (R²)', fontsize=12)
ax.set_title('Regulatory Complexity vs Model Performance', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('images/supp_fig2_regulators_vs_r2.svg', dpi=300, bbox_inches='tight', format = 'svg',)
plt.show()

In [ ]:
#mad vs r^2
fig, ax = plt.subplots(figsize=(10, 7))

hb = ax.hexbin(
    merged['MAD'],
    merged['Total_Explained_Variance'],
    gridsize=40, cmap='YlOrRd', mincnt=1, alpha=0.9,
    extent=[0, 3, -0.7, 1.1]
)
cb = plt.colorbar(hb, ax=ax, label='Number of genes')

ax.axhline(y=0.7, color='blue', linestyle='--', linewidth=1.5, label='R²=0.7 threshold')
ax.axhline(y=0.0, color='red', linestyle='--', linewidth=1.5, label='R²=0 threshold')

ax.set_xlim(0, 3)
ax.set_ylim(-0.7, 1.1)

ax.set_xlabel('MAD (Median Absolute Deviation)', fontsize=12)
ax.set_ylabel('Total Explained Variance (R²)', fontsize=12)
ax.set_title('Expression Variability vs Model Performance',
             fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=10, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('images/supp_figA_MAD_density.svg', dpi=300, bbox_inches='tight', format = 'svg',)
plt.show()

In [ ]:
#imod membership vs performance
df_plot = merged[merged['Total_Explained_Variance'] >= -1]

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(df_plot['num_iM'], df_plot['Total_Explained_Variance'],
                     c=df_plot['log_TPM_median'], cmap='coolwarm', alpha=0.5, s=15)
plt.colorbar(scatter, ax=ax, label='Median log TPM')
ax.axhline(y=0.7, color='orange', linestyle='--', linewidth=1.5, label='R²=0.7 threshold')
ax.axhline(y=0.0, color='red', linestyle='--', linewidth=1.5, label='R²=0 threshold')

corr = merged['num_iM'].corr(merged['Total_Explained_Variance'])
ax.text(0.98, 0.05, f'Pearson r = {corr:.3f}',
        transform=ax.transAxes, fontsize=10, ha='right', color='gray')

ax.set_xlabel('Number of iModulons Gene Belongs To', fontsize=12)
ax.set_ylabel('Total Explained Variance (R²)', fontsize=12)
ax.set_title('iModulon Membership vs Model Performance\nColored by Expression Level', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('images/supp_figC_iModulons_vs_R2.svg', dpi=300, bbox_inches='tight', format = 'svg',)
plt.show()

In [ ]:
# top 15 imod activity across 20 conditions
imodulon_sizes = ica_data.M_binarized.sum(axis=0).sort_values(ascending=False)
top_iM = imodulon_sizes.head(15).index.tolist()
A_subset = ica_data.A.loc[top_iM]


top_conditions = A_subset.var(axis=0).nlargest(20).index
A_plot = A_subset[top_conditions]

condition_labels = ica_data.sample_table.loc[top_conditions, 'condition']

fig, ax_heat = plt.subplots(figsize=(14, 8))

sns.heatmap(
    A_plot,
    xticklabels=condition_labels,
    yticklabels=top_iM,
    cmap='RdBu_r',
    center=0,
    ax=ax_heat,
    linewidths=0.3,
    cbar_kws={'label': 'iModulon Activity Score'}
)

ax_heat.set_xticklabels(ax_heat.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax_heat.set_yticklabels(ax_heat.get_yticklabels(), rotation=0, fontsize=9)
ax_heat.set_title('iModulon Activity Across Conditions\nTop 15 iModulons × Top 20 Variable Conditions',
                  fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('images/supp_fig_heatmap.svg', dpi=300, bbox_inches='tight', format = 'svg',)
plt.show()

In [ ]:
#old mad plot
df_plot = merged[(merged['MAD'] <= 3) & (merged['Total_Explained_Variance'] >= -1)]

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(df_plot['MAD'], df_plot['Total_Explained_Variance'],
                     c=df_plot['num_iM'], cmap='viridis', alpha=0.5, s=15)
plt.colorbar(scatter, ax=ax, label='Number of iModulons')
ax.axhline(y=0.7, color='orange', linestyle='--', linewidth=1.5, label='R²=0.7 threshold')
ax.axhline(y=0.0, color='red', linestyle='--', linewidth=1.5, label='R²=0 threshold')
ax.set_xlabel('MAD', fontsize=12)
ax.set_ylabel('Total Explained Variance (R²)', fontsize=12)
ax.set_title('Expression Variability vs Model Performance\nColored by Number of iModulons', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('images/supp_figA_MAD_vs_R2.svg', dpi=300, bbox_inches='tight', format = 'svg',)
plt.show()